<a target="_blank" href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_DEMO_SuperAnimal.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/>
</a>

# DeepLabCut SuperAnimal 模型

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1616492373700-PGOAC72IOB6AUE47VTJX/ke17ZwdGBToddI8pDm48kB8JrdUaZR-OSkKLqWQPp_YUqsxRUqqbr1mOJYKfIPR7LoDQ9mXPOjoJoqy81S2I8N_N4V1vUb5AoIIIbLZhVYwL8IeDg6_3B-BRuF4nNrNcQkVuAT7tdErd0wQFEGFSnBqyW03PFN2MN6T6ry5cmXqqA9xITfsbVGDrg_goIDasRCalqV8R3606BuxERAtDaQ/modelzoo.png?format=1000w)

http://modelzoo.deeplabcut.org

您可以使用此 Notebook，利用我们模型库中预训练好的网络来分析视频——**无需在本地安装 DeepLabCut**！

- **您需要准备什么：** 您喜欢的狗、猫、人等的视频：点击此处查看当前可用模型的列表：http://modelzoo.deeplabcut.org

- **操作步骤：** (1) 在右上角，点击 "CONNECT"。然后，只需对下面每个单元格点击运行（播放图标）并遵循说明即可！

- **注意，如果您的性能不尽如人意：** 首先请检查 `labeled_video` 参数（例如 "pcutoff" 可能会影响视频绘图的阈值）——请参阅此 Notebook 的末尾部分。
- 您也可以在您自己的本地项目中使用这些模型。请务必引用模型的对应论文，例如 [Ye et al. 2024](https://arxiv.org/abs/2203.07436) 🎉



## **开始吧：在 COLAB 中安装 DeepLabCut：**

*另外，请确保您已连接到 GPU：前往菜单，点击 Runtime > Change Runtime Type > 选择 "GPU"*

In [ ]:
!pip install --pre deeplabcut

## 请务必先点击上方输出中的 “restart runtime”（重启运行时）！

In [ ]:
from pathlib import Path

import deeplabcut

## 请选择一个您想在 SuperAnimal-X 上运行的视频：

In [ ]:
from google.colab import files

uploaded = files.upload()
for filepath, content in uploaded.items():
  print(f'User uploaded file "{filepath}" with length {len(content)} bytes')

video_path = Path(filepath).resolve()

# If this cell fails (e.g., when using Safari in place of Google Chrome),
# manually upload your video via the Files menu to the left
# and define `video_path` yourself with right click > copy path on the video.

## 接下来，选择你想使用的模型：`Quadruped` 或 `TopViewMouse`
- 更多关于这些模型的详细信息，请参阅 http://modelzoo.deeplabcut.org/
- `pcutoff` 参数仅用于可视化目的。这意味着只有关键点（keypoints）的置信度值高于你设定的阈值时才会被显示。0 表示模型的置信度低，1 表示模型的置信度完美。

In [ ]:
superanimal_name = "superanimal_quadruped" #@param ["superanimal_topviewmouse", "superanimal_quadruped"]
model_name = "hrnet_w32" #@param ["hrnet_w32", "resnet_50"]
detector_name = "fasterrcnn_resnet50_fpn_v2" #@param ["fasterrcnn_resnet50_fpn_v2", "fasterrcnn_mobilenet_v3_large_fpn"]
pcutoff = 0.15 #@param {type:"slider", min:0, max:1, step:0.05}

好的，我们开始吧！ 🐭🦓🐻

In [ ]:
videotype = video_path.suffix
scale_list = []

deeplabcut.video_inference_superanimal(
    [video_path],
    superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    videotype=videotype,
    video_adapt=True,
    scale_list=scale_list,
    pcutoff=pcutoff,
)

## 在 Colab 中查看视频：
- 否则，您可以从屏幕左侧下载并查看视频！它的文件名将以 `_labeled.mp4` 结尾。
- 如果您的数据效果不如预期，请考虑以下操作：使用您自己的数据对我们的模型进行微调 (fine-tuning)，更改 `pcutoff` 参数，或更改 `scale-range` 参数。
  (请选择比您的视频图像输入尺寸更小和更大的值)。有关更多详细信息，请参阅我们的代码仓库 (repo)。

In [ ]:
from base64 import b64encode
from IPython.display import HTML
import glob

# Get the parent directory and stem (filename without extension)
directory = video_path.parent
basename = video_path.stem

# Build the pattern
# This uses '*' to allow for any characters between the fixed parts
pattern = f"{basename}*{superanimal_name}*{detector_name}*{model_name}*_labeled_after_adapt.mp4"

# Search for matching files
matches = list(directory.glob(pattern))

# Choose the first match if it exists
labeled_video_path = matches[0] if matches else None

view_video = open(labeled_video_path, "rb").read()

data_url = "data:video/mp4;base64," + b64encode(view_video).decode()
HTML("""
<video width=600 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)